# 股票收益预测

将当日八个风格暴露、八个 Alpha 暴露与对应的预测因子收益相乘并求和，生成次日开盘至后日开盘的预测收益 `return_t2_pre`。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

## 文件路径与参数

自动向上定位项目根目录，使用项目相对路径读写文件；内核工作目录应位于本项目内。下方同时定义本步骤使用的参数。


In [ ]:
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'environment.yml').is_file() and (path / 'src').is_dir()
)

RETURN_START = pd.Timestamp("20230101")

PANEL = PROJECT_ROOT / "data/processed/panel.parquet"
ALPHA_FACTOR_EXPOSURE = PROJECT_ROOT / "src/factor_exposure/alpha_factor/alpha_factor_exposure.parquet"
STYLE_FACTOR_EXPOSURE = PROJECT_ROOT / "src/factor_exposure/style_factor/style_factor_exposure.parquet"
FACTOR_RETURN = PROJECT_ROOT / "src/factor_return/factor_return.parquet"
STOCK_RETURN_OUTPUT = PROJECT_ROOT / "src/stock_return/stock_return.parquet"


## 读取暴露与因子收益预测

使用已生成的 Alpha 暴露、风格暴露和因子收益合并表。所有数据按日期、股票代码进行匹配。


In [ ]:
panel = pd.read_parquet(PANEL)
alpha_factor_exposure = pd.read_parquet(ALPHA_FACTOR_EXPOSURE)
style_factor_exposure = pd.read_parquet(STYLE_FACTOR_EXPOSURE)
factor_return = pd.read_parquet(FACTOR_RETURN)

## 对齐暴露并统一方向

选取与因子收益模型相同的八个 Alpha，反向处理同样四个因子，再合并八个风格暴露。明确因子列顺序，保证与预测系数逐列对应。


In [ ]:
merge_keys = ['trade_date', 'ts_code']
alpha_cols = ['alpha42', 'alpha80', 'alpha112', 'alpha177',
              'alpha1', 'alpha110', 'alpha132', 'alpha12']
style_cols = ['size', 'mom', 'vol1', 'vol2', 'liq1', 'liq2', 'beta', 'value']
factor_cols = style_cols + alpha_cols
factor_pred_cols = [f'{col}_pre' for col in factor_cols]

stock_return = alpha_factor_exposure.loc[
    alpha_factor_exposure['trade_date'] >= RETURN_START,
    merge_keys + alpha_cols
].copy().reset_index(drop=True)

reverse_cols = ['alpha80', 'alpha112', 'alpha177', 'alpha110']
stock_return[reverse_cols] = -stock_return[reverse_cols]

stock_return = stock_return.merge(
    style_factor_exposure[merge_keys + style_cols],
    on=merge_keys, how='left', sort=False, validate='one_to_one'
)

## 合成股票预测收益

按信号日匹配 16 个 `_pre` 列：

$$
\widehat r_{i,t+2}=\sum_{j=1}^{16}x_{i,j,t}\widehat f_{j,t}.
$$

只使用预测因子收益，不加入回归截距。任一必要输入缺失时，合成结果为缺失。信号日 $t$ 的结果用于 $t+1$ 日开盘决策。


In [ ]:
return_prediction_inputs = stock_return[['trade_date'] + factor_cols].merge(
    factor_return[['trade_date'] + factor_pred_cols],
    on='trade_date', how='left', sort=False, validate='many_to_one'
)

exposures = return_prediction_inputs[factor_cols].to_numpy()
predicted_factor_returns = return_prediction_inputs[factor_pred_cols].to_numpy()
stock_return['return_t2_pre'] = (
    exposures * predicted_factor_returns
).sum(axis=1)

## 整理预测结果

只保留 `ts_code`、`trade_date`、`return_t2_pre`，预览后保存，避免将实际因子收益误带入回测输入。


In [ ]:
stock_return = stock_return[['ts_code', 'trade_date', 'return_t2_pre']]

In [ ]:
stock_return.head()

## 保存股票预测

写入 `STOCK_RETURN_OUTPUT`。回测按上一交易日匹配本表，并结合执行日状态筛选股票。


In [ ]:
stock_return.to_parquet(STOCK_RETURN_OUTPUT, index=False)